In [ ]:
%reload_ext autoreload
%autoreload 2

import json
from pprint import pprint
import numpy as np
import jax
import jax.numpy as jnp
import healpy as hp

from fpp.models.np_model import NPModel
from fpp.utils.posterior import multi_corner

# 1. Simulate data

We use `NPModel.simulate` to generate a mock dataset from a known truth dictionary,
then fit the same model back to it.

In [ ]:
m = NPModel()

truth_dict = json.load(open("example_truth.json", "r"))
pprint(truth_dict)
sim_map = m.simulate(truth_dict)
data_in = jnp.array(sim_map, dtype=jnp.int32)

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({'text.usetex': False}) #comment out this line if you want to use LaTeX for the figure labels

plot_data = np.array(np.clip(data_in, 0.5, None))
plot_data[m.mask_roi] = np.nan
hp.cartview(plot_data, latra=[-25, 25], lonra=[-25, 25], title="Simulated Map (ROI)", unit="Counts", norm='log')

# 2. Fit

We create a fresh model with the simulated data (so that `k_max` is set correctly),
then run SVI or HMC.

In [ ]:
m = NPModel(data=data_in)

In [ ]:
# SVI fit
m.fit_svi(
    data=data_in,
    guide='iaf', num_flows=5, hidden_dims=[128, 128],
    lr=3e-4, n_steps=5000,
)
samples_svi = m.get_svi_samples(num_samples=10000)

In [ ]:
# NUTS fit
mcmc = m.run_nuts(
    data=data_in,
    num_chains=4, num_warmup=500,
    num_samples=5000, step_size=0.05,
)
samples_hmc = m.expand_samples(mcmc.get_samples())

# 3. Posterior

Corner plot comparing SVI vs NUTS, with the simulation truth overlaid.

In [ ]:
plot_keys = ['S_pib', 'S_ics', 'S_iso', 'S_bub', 'S_nfw', 'Sps_dsk', 'n1_dsk', 'n2_dsk', 'n3_dsk', 'sb1_dsk', 'lambdas_dsk']
latex_labels = [
    r'$S_\mathrm{pib}$', r'$S_\mathrm{ics}$', r'$S_\mathrm{iso}$', r'$S_\mathrm{bub}$', r'$S_\mathrm{nfw}$', r'$S^\mathrm{ps}_\mathrm{dsk}$',
    r'$n_1^\mathrm{dsk}$', r'$n_2^\mathrm{dsk}$', r'$n_3^\mathrm{dsk}$', r'$S_{b,1}^\mathrm{dsk}$', r'$\lambda_s^\mathrm{dsk}$'
]

multi_corner(
    {'hmc': samples_hmc, 'svi': samples_svi},
    plot_keys,
    labels=latex_labels,
    point_est={k: truth_dict[k] for k in labels},
    colors_dict={'hmc': 'k', 'svi': 'C0'},
    legend_dict={'hmc': r'$\text{HMC}$', 'svi': r'$\text{SVI}$'},
    legend_loc=(0.15, 0.94),
    label_kwargs={"fontsize": 30},
)